In [0]:
# Databricks notebook source
# Task 03: Generar embeddings y guardar en la misma Delta Table

# COMMAND ----------
# Configuracion base (constantes)
UC_CATALOG = "parts"
UC_SCHEMA = "bronze"

# COMMAND ----------
# Config file (DBFS)
# Ejemplo: dbfs:/config/embeddings.json
CONFIG_PATH = "dbfs:/config/embeddings.json"

# COMMAND ----------
import json
import time
import requests
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType, TimestampType
import logging

logger = logging.getLogger("embeddings")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

def log(msg: str):
    ts = datetime.utcnow().isoformat()
    line = f"{ts} | {msg}"
    print(line)
    logger.info(line)

def get_api_key(cfg):
    scope = (cfg.get("secret_scope") or "").strip()
    key = (cfg.get("secret_key") or "").strip()
    if scope and key:
        return dbutils.secrets.get(scope=scope, key=key)
    # fallback a variable de entorno
    import os
    return os.environ.get("OPENAI_API_KEY", "")

def load_config(path: str):
    try:
        raw = dbutils.fs.head(path, 1_000_000)
    except Exception as e:
        raise ValueError(f"No se pudo leer config en {path}") from e
    try:
        return json.loads(raw)
    except Exception as e:
        raise ValueError(f"Config JSON invalido en {path}") from e

def embed_texts(texts, model, dimensions=None, max_retries=3):
    # Azure OpenAI endpoint
    if not azure_endpoint or not azure_deployment:
        raise ValueError("Faltan azure_endpoint o azure_deployment en config.")
    url = f"{azure_endpoint}/openai/deployments/{azure_deployment}/embeddings?api-version={azure_api_version}"
    headers = {
        "api-key": api_key,
        "Content-Type": "application/json",
    }
    payload = {"input": texts}
    # Azure ignora 'model' cuando usas deployment, pero lo mantenemos como metadato
    if dimensions:
        payload["dimensions"] = int(dimensions)

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(url, headers=headers, data=json.dumps(payload), timeout=60)
            if resp.status_code >= 400:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")
            data = resp.json()
            return [d["embedding"] for d in data["data"]]
        except Exception as e:
            last_err = e
            time.sleep(2 * attempt)
    raise RuntimeError(f"OpenAI embeddings failed after {max_retries} attempts: {last_err}")

def build_embedding_text(df):
    preferred_cols = [
        "part_number",
        "category",
        "subcategory",
        "description",
        "manufacturer_brand",
        "material",
        "specifications_met",
        "notes",
        "__text",
    ]
    pieces = []
    for c in preferred_cols:
        if c in df.columns:
            pieces.append(
                F.when(
                    F.col(c).isNotNull(),
                    F.concat(F.lit(f"{c}: "), F.col(c).cast("string"))
                )
            )
    if not pieces:
        raise ValueError("No hay columnas disponibles para construir embedding_text.")
    return df.withColumn("embedding_text", F.concat_ws(" | ", *pieces))

# COMMAND ----------
cfg = load_config(CONFIG_PATH)
api_key = get_api_key(cfg)
if not api_key:
    raise ValueError("No se encontro OPENAI_API_KEY. Configura secret_scope/secret_key o variable de entorno.")

table_pattern = (cfg.get("table_pattern") or "*").strip()
if table_pattern == "%":
    table_pattern = "*"
model = (cfg.get("model") or "text-embedding-3-small").strip()
catalog = (cfg.get("catalog") or UC_CATALOG).strip()
schema = (cfg.get("schema") or UC_SCHEMA).strip()
azure_endpoint = (cfg.get("azure_endpoint") or "").strip().rstrip("/")
azure_deployment = (cfg.get("azure_deployment") or "").strip()
azure_api_version = (cfg.get("azure_api_version") or "2024-02-15-preview").strip()
dimensions = str(cfg.get("dimensions") or "").strip()
batch_size = int(cfg.get("batch_size") or 100)
only_missing = bool(cfg.get("only_missing", True))

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# Forzar catalog/schema
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# COMMAND ----------
# Obtener tablas a procesar
current_catalog = spark.sql("SELECT current_catalog() AS c").collect()[0]["c"]
current_schema = spark.sql("SELECT current_schema() AS s").collect()[0]["s"]
log(f"Catalog/Schema actual: {current_catalog}.{current_schema}")

tables = (
    spark.sql(f"SHOW TABLES IN {catalog}.{schema} LIKE '{table_pattern}'")
    .select("tableName")
    .collect()
)
table_names = [f"{catalog}.{schema}.{r['tableName']}" for r in tables]

if not table_names:
    log(f"WARNING: No hay tablas en {catalog}.{schema}. Intentando fallback a {current_catalog}.{current_schema}.")
    tables = (
        spark.sql(f"SHOW TABLES IN {current_schema} LIKE '{table_pattern}'")
        .select("tableName")
        .collect()
    )
    table_names = [f"{current_catalog}.{current_schema}.{r['tableName']}" for r in tables]

if not table_names:
    raise ValueError(
        f"No se encontraron tablas con pattern '{table_pattern}' en "
        f"{catalog}.{schema} ni en {current_catalog}.{current_schema}."
    )

log(f"Tablas a procesar: {table_names}")

# COMMAND ----------
for table_name in table_names:
    log(f"Procesando tabla: {table_name}")
    df = spark.table(table_name)

    if "part_number" not in df.columns:
        log(f"SKIP: {table_name} no tiene 'part_number'")
        continue

    df = build_embedding_text(df)
    df = df.filter(F.col("part_number").isNotNull())

    if only_missing and "embedding" in df.columns:
        df = df.filter(F.col("embedding").isNull())

    # Evitar re-embeddings duplicados en el batch
    df = df.dropDuplicates(["part_number"])

    # Recolectar iterativamente para no cargar todo en memoria
    rows = []
    batch_texts = []
    batch_keys = []

    def flush_batch():
        if not batch_texts:
            return
        embeds = embed_texts(batch_texts, model, dimensions if dimensions else None)
        for k, t, e in zip(batch_keys, batch_texts, embeds):
            rows.append((k, t, e))
        batch_texts.clear()
        batch_keys.clear()

    for r in df.select("part_number", "embedding_text").toLocalIterator():
        batch_keys.append(r["part_number"])
        batch_texts.append(r["embedding_text"])
        if len(batch_texts) >= batch_size:
            flush_batch()

    flush_batch()

    if not rows:
        log(f"Sin filas para embeddings en {table_name}")
        continue

    now_ts = datetime.utcnow()
    embed_schema = StructType([
        StructField("part_number", StringType(), False),
        StructField("embedding_text", StringType(), True),
        StructField("embedding", ArrayType(FloatType()), True),
        StructField("embedding_model", StringType(), True),
        StructField("embedding_dim", StringType(), True),
        StructField("embedding_ts", TimestampType(), True),
    ])

    embed_df = spark.createDataFrame(
        [(k, t, e, model, str(len(e)), now_ts) for k, t, e in rows],
        schema=embed_schema
    )

    delta_tbl = DeltaTable.forName(spark, table_name)
    merge_cond = "t.part_number = s.part_number"
    if only_missing and "embedding" in df.columns:
        update_cond = "t.embedding IS NULL"
        (delta_tbl.alias("t")
            .merge(embed_df.alias("s"), merge_cond)
            .whenMatchedUpdate(condition=update_cond, set={
                "embedding": "s.embedding",
                "embedding_model": "s.embedding_model",
                "embedding_dim": "s.embedding_dim",
                "embedding_text": "s.embedding_text",
                "embedding_ts": "s.embedding_ts",
            })
            .whenNotMatchedInsert(values={
                "part_number": "s.part_number",
                "embedding": "s.embedding",
                "embedding_model": "s.embedding_model",
                "embedding_dim": "s.embedding_dim",
                "embedding_text": "s.embedding_text",
                "embedding_ts": "s.embedding_ts",
            })
            .execute())
    else:
        (delta_tbl.alias("t")
            .merge(embed_df.alias("s"), merge_cond)
            .whenMatchedUpdate(set={
                "embedding": "s.embedding",
                "embedding_model": "s.embedding_model",
                "embedding_dim": "s.embedding_dim",
                "embedding_text": "s.embedding_text",
                "embedding_ts": "s.embedding_ts",
            })
            .whenNotMatchedInsert(values={
                "part_number": "s.part_number",
                "embedding": "s.embedding",
                "embedding_model": "s.embedding_model",
                "embedding_dim": "s.embedding_dim",
                "embedding_text": "s.embedding_text",
                "embedding_ts": "s.embedding_ts",
            })
            .execute())

    log(f"Embeddings upserted en {table_name}: {len(rows)} filas")

log("Task 03 completed.")
